# 2. 主题聚类 & 向量抽取
## 向量聚类
用 KMeans（或其它密度聚类）对上面得到的句向量做聚类，每个簇就对应一个“语料主题”。
## 主题向量
对每个簇内所有句向量取加权平均（权重可以是 TF-IDF 分数或簇成员的密度分数），得到一个簇中心向量，代表该主题。

In [44]:
from sklearn.cluster import KMeans
import numpy as np

# 1) 假设 embeddings 已经是 [n_texts, D] 的 SBERT 嵌入矩阵
# 2) 指定主题数
n_topics = 10

# 3) 用 KMeans 聚类
kmeans = KMeans(n_clusters=n_topics, random_state=42)
labels = kmeans.fit_predict(embeddings)         # 每条文本分到 0~9 号簇
centers = kmeans.cluster_centers_               # shape: (10, D)

# 4)（可选）单位向量归一化，方便后面余弦相似度计算
norms = np.linalg.norm(centers, axis=1, keepdims=True)
topic_vectors = centers / norms                 # 10×D 的主题向量

# 5) 快速检查每个簇的大小
from collections import Counter
print("各主题簇大小：", Counter(labels))

# 6) 打印确认
print(f"已生成 {n_topics} 个主题向量，每个维度是 {topic_vectors.shape[1]}")


各主题簇大小： Counter({np.int32(1): 76, np.int32(2): 70, np.int32(5): 64, np.int32(6): 60, np.int32(9): 53, np.int32(4): 47, np.int32(3): 35, np.int32(7): 30, np.int32(8): 16, np.int32(0): 12})
已生成 10 个主题向量，每个维度是 768


# 2.1 KMeans 聚类给你的是“哪条文本属于哪个簇”＋“每个簇的中心向量”，但它并不告诉你“这个簇在讲什么”。簇号（0～9）只是编号，并没有语义。要把它变成人类能读得懂的“主题”，必须做“簇可解释化”——也就是给每个簇取一个能反映其语义的标签或摘要。
尝试给簇“贴标签”的思路：

# 2.1a 方法一：传统TF-IDF

In [55]:
# n_topics 的主题是什么呢？我想知道
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 1. 用 TF-IDF 向量化所有文本
vectorizer = TfidfVectorizer(
    max_df=0.8,        # 在 ≥80% 文档中出现的词扔掉
    min_df=3,          # 在 <3 篇文档中出现的词也扔掉
    stop_words='english'  # 英文停用词
)
X_tfidf = vectorizer.fit_transform(documents)
features = vectorizer.get_feature_names_out()

# 2. 为每个簇计算「累计 TF-IDF」并取 Top-N 关键词
n_top = 10
topic_keywords = {}
for topic in range(n_topics):
    # 找到属于这个簇的文本索引
    idx = np.where(labels == topic)[0]
    if len(idx) == 0:
        topic_keywords[topic] = []
        continue

    # 把这些文档的 TF-IDF 加总（得到每个词在这个簇的总体重要度）
    tfidf_sum = X_tfidf[idx].sum(axis=0)            # 1×V 矩阵
    tfidf_sum = np.asarray(tfidf_sum).ravel()       # 转成长度 V 的向量

    # 取累计 TF-IDF 最大的前 n_top 个词
    top_indices = tfidf_sum.argsort()[::-1][:n_top]
    top_terms   = [features[i] for i in top_indices]

    topic_keywords[topic] = top_terms

# 3. 打印每个簇的关键词
for topic, kws in topic_keywords.items():
    print(f"主题 {topic} （簇大小 {np.sum(labels==topic)} 段文段）:")
    print("  ", ", ".join(kws))
    print()

主题 0 （簇大小 12 段文段）:
   volume, fair, strong, building, onenyc, 2050, york, healthy, reliable, climate

主题 1 （簇大小 76 段文段）:
   york, climate, yorkers, health, 2050, global, housing, strong, communities, infrastructure

主题 2 （簇大小 70 段文段）:
   bus, community, data, access, service, projects, dot, including, york, fleet

主题 3 （簇大小 35 段文段）:
   conversation, tag, media, join, 2017, social, 2010, onenyc, 2018, 2000

主题 4 （簇大小 47 段文段）:
   climate, students, yorkers, york, health, programs, jobs, school, ensure, create

主题 5 （簇大小 64 段文段）:
   mayor, source, york, april, 2018, 2017, onenyc, advisor, williams, fuleihan

主题 6 （簇大小 60 段文段）:
   health, yorkers, public, york, neighborhoods, mental, program, community, services, care

主题 7 （簇大小 30 段文段）:
   onenyc, funded, gov, learn, nyc, tag, mayor, media, fair, partially

主题 8 （簇大小 16 段文段）:
   yorkers, health, promote, infrastructure, democracy, advance, climate, equity, strengthen, make

主题 9 （簇大小 53 段文段）:
   health, york, climate, housing, yorkers, em

# 2.1b 方法二：簇内“最近邻”代表句

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

for topic_id in range(n_topics):
    idx = np.where(labels == topic_id)[0]            # 属于该簇的句子索引
    center = topic_vectors[topic_id]                  # 或 centers[topic_id]，取决你存哪里
    sims = cosine_similarity(embeddings[idx],
                             center.reshape(1, -1)).ravel()
    topk = sims.argsort()[::-1][:5]                   # 相似度最高的 5 条
    repr_sentences = [documents[idx[i]] for i in topk]
    print(f"簇 {topic_id} 的代表句：")
    for sent in repr_sentences:
        print("  -", sent)
    print()

簇 0 的代表句：
  - ONENYC 2050 IS A STRATEGY TO SECURE OUR CITY’S FUTURE AGAINST THE CHALLENGES OF TODAY AND TOMORROW. WITH BOLD ACTIONS TO CONFRONT OUR CLIMATE CRISIS, ACHIEVE EQUITY, AND STRENGTHEN OUR DEMOCRACY, WE ARE BUILDING A STRONG AND FAIR CITY. JOIN US. OneNYC 2050  OneNYC 2050  OneNYC 2050  BUILDING A STRONG AND FAIR CITY  THRIVING NEIGHBORHOODS  HEALTHY LIVES  VOLUME 4 OF 9  VOLUME 5 OF 9  New York City will grow and diversify its economy so that it creates opportunity for all, safeguards the American dream and addresses the racial wealth gap.  New York City will foster communities that have safe and affordable housing and are wellserved by parks, cultural resources, and shared spaces.  New York City will reduce inequities in health outcomes by addressing their root causes in residents’ daily lives, guaranteeing health care, and facilitating both healthy lifestyles and a healthy physical environment.  OneNYC 2050  OneNYC 2050  OneNYC 2050  A LIVABLE CLIMATE  EFFICIENT MOBILITY  

# 2.1c 方法三：簇中心“总结”标签（借助 LLM）

用方法二拿到每个簇的 5–10 条代表句；

构造 prompt，问 ChatGPT/其他 LLM：“下面是几个句子，它们好像都跟同一个主题有关，请帮我用一句话总结这个主题。”

LLM 返回的那句话，就可以直接当作“主题名称”或“标签”了。

In [47]:
import os
import openai

# 1. 配置你的 OpenAI API Key
openai.api_key = os.getenv("OPENAI_API_KEY")  # 或者直接写在这里：openai.api_key = "你的密钥"

def summarize_cluster(repr_sentences, model="gpt-3.5-turbo"):
    """
    给定一个簇的代表句列表，调用 ChatCompletion 接口返回一句话的主题总结。
    """
    # 拼接代表句到一个文本段落
    examples = "\n".join(f"- {s}" for s in repr_sentences)
    prompt = (
        "下面是几个句子，它们好像都跟同一个主题有关：\n"
        f"{examples}\n\n"
        "请帮我用一句话总结这个主题，尽量简洁并直接给出主题名称。"
    )

    resp = openai.ChatCompletion.create(
        model=model,
        messages=[
            {"role": "system", "content": "你是一个擅长文本归纳的助手。"},
            {"role": "user",   "content": prompt}
        ],
        temperature=0.3,
        max_tokens=32,
        n=1,
    )
    # 提取接口返回的文本
    label = resp.choices[0].message.content.strip()
    return label

if __name__ == "__main__":
    # 假设已经有 cluster_reprs
    cluster_reprs = {
        0: ["Urban design shapes how we live in cities.",
            "The layout of public spaces affects community interaction.",
            "Design guidelines impact pedestrian flow and safety.",
            "Architectural style influences urban vitality.",
            "Green infrastructure is integrated into building design."],
        1: ["Transportation planning improves mobility.",
            "Public transit accessibility reduces traffic congestion.",
            "Bus routes and schedules are optimized for demand.",
            "Cycling lanes and pedestrian paths are expanded.",
            "Multimodal hubs connect metro, bus, and bike share."],
        # … 其余簇
    }

    # 对每个簇调用一下，打印出主题标签
    cluster_labels = {}
    for cid, reps in cluster_reprs.items():
        label = summarize_cluster(reps)
        print(f"簇 {cid} 的主题标签：{label}")
        cluster_labels[cid] = label

    # 如果需要，可以把结果保存到文件或进一步使用
    # e.g. import json; json.dump(cluster_labels, open("labels.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)


SyntaxError: invalid character '–' (U+2013) (3290076049.py, line 1)